# YOLOv11-n Mango Detection

This notebook fine-tunes and evaluates a YOLOv11-n object detector for single-class mango detection in orchard images.

**Final recorded test results**

| Metric | Result |
|---|---:|
| Precision | 0.936436 |
| Recall | 0.971182 |
| mAP@0.5 | 0.989359 |
| mAP@0.5:0.95 | 0.704345 |

The notebook has been reorganised for portfolio use. Student identifiers, assessment wording, and personal Google Drive paths have been removed.

## 1. Install dependencies

The original experiments used Ultralytics 8.4.52. Restart the runtime only when Colab requests it.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/JeraldBucud/yolov11-mango-detection/main/requirements.txt

## 2. Clone the repository in Google Colab

Skip this cell when running the notebook from a local clone.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/yolov11-mango-detection")
    if not repo_dir.exists():
        !git clone -q https://github.com/JeraldBucud/yolov11-mango-detection.git "$repo_dir"
    %cd /content/yolov11-mango-detection
else:
    repo_dir = Path.cwd()

print("Repository directory:", repo_dir.resolve())

## 3. Imports and runtime information

In [ ]:
from pathlib import Path
import os
import yaml
import pandas as pd
import torch

from ultralytics import YOLO
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configure the dataset

The full dataset is not committed to GitHub. Place the downloaded YOLO-format dataset at `data/mangoyolo`, or set the `MANGO_DATASET_DIR` environment variable.

For Google Colab, you may mount Google Drive and point the variable to a Drive folder.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_dataset = (
    Path("/content/drive/MyDrive/mangoyolo")
    if IN_COLAB
    else repo_dir / "data" / "mangoyolo"
)

DATASET_DIR = Path(os.getenv("MANGO_DATASET_DIR", default_dataset))
RUNS_DIR = repo_dir / "runs"
RUNTIME_YAML = RUNS_DIR / "mango_data_runtime.yaml"

print("Dataset directory:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

## 5. Validate the dataset structure

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_files(folder: Path, extensions: set[str]) -> int:
    if not folder.exists():
        return 0
    return sum(
        1 for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )

rows = []
for split in ("train", "valid", "test"):
    image_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"
    image_count = count_files(image_dir, IMAGE_EXTENSIONS)
    label_count = count_files(label_dir, {".txt"})
    rows.append({
        "Split": split,
        "Images": image_count,
        "Labels": label_count,
        "Counts match": image_count == label_count,
    })

dataset_summary = pd.DataFrame(rows)
dataset_summary

## 6. Create a runtime YOLO configuration

The runtime YAML uses an absolute dataset path, preventing machine-specific path errors.

In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "Dataset not found. Download it and update MANGO_DATASET_DIR before continuing."
    )

RUNS_DIR.mkdir(parents=True, exist_ok=True)

runtime_config = {
    "path": str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": {0: "mango"},
}

with RUNTIME_YAML.open("w", encoding="utf-8") as file:
    yaml.safe_dump(runtime_config, file, sort_keys=False)

print(RUNTIME_YAML.read_text(encoding="utf-8"))

## 7. Training experiments

Set `RUN_TRAINING = True` to repeat the experiments. Training can take significant time and requires the complete dataset.

In [ ]:
RUN_TRAINING = False

experiments = {
    "baseline": {
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
        "lr0": 0.001,
        "optimizer": "AdamW",
        "patience": 15,
        "name": "exp1_yolov11n_baseline",
    },
    "tuned": {
        "epochs": 100,
        "imgsz": 640,
        "batch": 16,
        "lr0": 0.0008,
        "optimizer": "AdamW",
        "patience": 20,
        "name": "exp2_yolov11n_tuned",
    },
}

if RUN_TRAINING:
    for experiment_name, settings in experiments.items():
        print(f"Starting {experiment_name} experiment")
        model = YOLO("yolo11n.pt")
        model.train(
            data=str(RUNTIME_YAML),
            project=str(RUNS_DIR),
            exist_ok=True,
            seed=42,
            pretrained=True,
            **settings,
        )
else:
    print("Training skipped. Set RUN_TRAINING = True to train the models.")

## 8. Validation comparison

These are the recorded validation results used for model selection.

In [ ]:
validation_results = pd.DataFrame([
    {
        "Experiment": "Baseline",
        "mAP@0.5": 0.988831,
        "mAP@0.5:0.95": 0.701359,
        "Precision": 0.962133,
        "Recall": 0.951333,
    },
    {
        "Experiment": "Tuned",
        "mAP@0.5": 0.990000,
        "mAP@0.5:0.95": 0.703000,
        "Precision": 0.966000,
        "Recall": 0.956000,
    },
])

validation_results.sort_values("mAP@0.5:0.95", ascending=False)

## 9. Evaluate the selected checkpoint

The included `models/best.pt` checkpoint is the selected tuned model.

In [ ]:
MODEL_PATH = repo_dir / "models" / "best.pt"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}")

model = YOLO(str(MODEL_PATH))

test_metrics = model.val(
    data=str(RUNTIME_YAML),
    split="test",
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name="final_test_evaluation",
    exist_ok=True,
)

test_results = pd.DataFrame([{
    "Precision": float(test_metrics.box.mp),
    "Recall": float(test_metrics.box.mr),
    "mAP@0.5": float(test_metrics.box.map50),
    "mAP@0.5:0.95": float(test_metrics.box.map),
}])

test_results

## 10. Run inference on an image or folder

In [ ]:
SOURCE = repo_dir / "data" / "samples" / "images"

prediction_results = model.predict(
    source=str(SOURCE),
    imgsz=640,
    conf=0.25,
    iou=0.50,
    save=True,
    project=str(RUNS_DIR),
    name="portfolio_predictions",
    exist_ok=True,
)

print("Saved predictions to:", RUNS_DIR / "portfolio_predictions")

## 11. Limitations and next steps

- Occluded, clustered, small, and poorly illuminated mangoes remain challenging.
- The model has been tested on one dataset and one held-out split.
- Future work should use additional orchards, cultivars, seasons, and lighting conditions.
- Useful extensions include mango counting, ripeness detection, disease detection, and web deployment.